### Installation
1. Run the first 2 cells
2. Restart runtime
3. Run the rest of the jupyter notebooks (do not run the first 2 cells again)

In [ ]:
!git clone -b main https://github.com/zcemycl/TF2DeepFloorplan.git
!pip install gdown
!pip install --upgrade --no-cache-dir gdown
!gdown https://drive.google.com/uc?id=1czUSFvk6Z49H-zRikTc67g2HUUz4imON
!unzip log.zip
!rm log.zip

In [ ]:
# gpu
# !cd TF2DeepFloorplan && pip install -e .[tfgpu]
# cpu
!cd TF2DeepFloorplan && pip install -e .[tfcpu]

### Main Script

In [ ]:
import tensorflow as tf
import sys
from dfp.net import *
from dfp.data import *
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
from argparse import Namespace
import os
import gc
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
from dfp.utils.rgb_ind_convertor import *
from dfp.utils.util import *
from dfp.utils.legend import *
from dfp.utils.settings import *
from dfp.deploy import *
print(tf.test.is_gpu_available())
print(tf.config.list_physical_devices('GPU'))

In [ ]:
img_path = './TF2DeepFloorplan/resources/30939153.jpg'
inp = mpimg.imread(img_path)
args = parse_args("--tomlfile ./TF2DeepFloorplan/docs/notebook.toml".split())
args = overwrite_args_with_toml(args)
args.image = img_path

In [ ]:
result = main(args)

In [ ]:
plt.subplot(1,2,1)
plt.imshow(inp); plt.xticks([]); plt.yticks([]);
plt.subplot(1,2,2)
plt.imshow(result); plt.xticks([]); plt.yticks([]);

## Breakdown of postprocessing (step by step)

In [ ]:
model,img,shp = init(args)
logits_cw,logits_r = predict(model,img,shp)

In [ ]:
logits_r = tf.image.resize(logits_r,shp[:2])
logits_cw = tf.image.resize(logits_cw,shp[:2])
r = convert_one_hot_to_image(logits_r)[0].numpy()
cw = convert_one_hot_to_image(logits_cw)[0].numpy()
plt.subplot(1,2,1)
plt.imshow(r.squeeze()); plt.xticks([]); plt.yticks([]);
plt.subplot(1,2,2)
plt.imshow(cw.squeeze()); plt.xticks([]); plt.yticks([]);

In [ ]:
r_color,cw_color = colorize(r.squeeze(),cw.squeeze())
plt.subplot(1,2,1)
plt.imshow(r_color); plt.xticks([]); plt.yticks([]);
plt.subplot(1,2,2)
plt.imshow(cw_color); plt.xticks([]); plt.yticks([]);

In [ ]:
newr,newcw = post_process(r,cw,shp)
plt.subplot(1,2,1)
plt.imshow(newr.squeeze()); plt.xticks([]); plt.yticks([]);
plt.subplot(1,2,2)
plt.imshow(newcw.squeeze()); plt.xticks([]); plt.yticks([]);

In [ ]:
newr_color,newcw_color = colorize(newr.squeeze(),newcw.squeeze())
plt.subplot(1,2,1)
plt.imshow(newr_color); plt.xticks([]); plt.yticks([]);
plt.subplot(1,2,2)
plt.imshow(newcw_color); plt.xticks([]); plt.yticks([]);

In [ ]:
plt.imshow(newr_color+newcw_color); plt.xticks([]); plt.yticks([]);

In [ ]:
over255 = lambda x: [p/255 for p in x]
colors2 = [over255(rgb) for rgb in list(floorplan_fuse_map.values())]
colors = ["background", "closet", "bathroom",
          "living room\nkitchen\ndining room",
          "bedroom","hall","balcony","not used","not used",
          "door/window","wall"]
f = lambda m,c: plt.plot([],[],marker=m, color=c, ls="none")[0]
handles = [f("s", colors2[i]) for i in range(len(colors))]
labels = colors
legend = plt.legend(handles, labels, loc=3,framealpha=1, frameon=True)

fig  = legend.figure
fig.canvas.draw()
plt.xticks([]); plt.yticks([]);


In [1]:
%pip install --quiet pymupdf-layout

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 47.0 MB/s eta 0:00:00


In [3]:
%pip install --quiet pymupdf4llm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.2/78.2 kB 8.8 MB/s eta 0:00:00


In [1]:
import pymupdf.layout
import pymupdf4llm

doc = pymupdf.open("/content/ERS-EX-ARQ-014-PLANTA_BAIXA_PAVIMENTO_TÉRREO_BLOCO_B-C_R04.pdf")

In [ ]:
md = pymupdf4llm.to_markdown(doc)

In [5]:
pymupdf4llm.use_layout(True)

In [6]:
text = pymupdf4llm.to_text(doc)

In [7]:
def parse_info_block(lines):
    """
    Associa campos a valores em blocos de informações, considerando que cada campo é seguido do valor.
    """
    campos = {
        "RESPONSÁVEL TÉCNICO EXECUÇÃO",
        "PROPRIETÁRIO(s)",
        "CONTEÚDO",
        "ENDEREÇO DA OBRA",
        "DATA",
        "ESCALA:",
        "DESENHO",
        "RESPONSÁVEL TÉCNICO PROJETO",
        "ASSINATURA",
        "PROJETO EXECUTIVO"
    }
    resultado = {}
    i = 0
    while i < len(lines):
        linha = lines[i].strip()
        if linha in campos:
            # ASSINATURA pode se repetir, então use uma lista
            if linha == "ASSINATURA":
                resultado.setdefault("ASSINATURA", []).append(lines[i+1].strip() if i+1 < len(lines) else "")
                i += 2
                continue
            # Campo normal
            valor = lines[i+1].strip() if i+1 < len(lines) else ""
            resultado[linha.rstrip(":")] = valor
            i += 2
        else:
            i += 1
    return resultado

# Exemplo de uso:
linhas = [
    "RESPONSÁVEL TÉCNICO EXECUÇÃO",
    "PROPRIETÁRIO(s)",
    "CONTEÚDO",
    "ENDEREÇO DA OBRA",
    "DATA",
    "ESCALA:",
    "DESENHO",
    "RESPONSÁVEL TÉCNICO PROJETO",
    "ASSINATURA",
    "ASSINATURA",
    "ASSINATURA",
    "PROJETO EXECUTIVO",
    "Como indicado",
    "18/03/2022",
    "FAGNER VASCO",
    "14-39",
    "PLANTA BAIXA PAVIMENTO TÉRREO BLOCO B-C",
    "SANTIN HOME CLUB",
    "Fábio Silva",
    "ARQUITETO E URBANISTA | CAU - A131393-2",
    "CBA Empreendimentos Imobiliários",
    "CPF/CNPJ - 22.407.836./0001-48",
    "Rua São Benedito, Serraria - São José",
    "_"
]

print(parse_info_block(linhas))

{'RESPONSÁVEL TÉCNICO EXECUÇÃO': 'PROPRIETÁRIO(s)', 'CONTEÚDO': 'ENDEREÇO DA OBRA', 'DATA': 'ESCALA:', 'DESENHO': 'RESPONSÁVEL TÉCNICO PROJETO', 'ASSINATURA': ['ASSINATURA', 'PROJETO EXECUTIVO']}


In [1]:
!pip install sentence-transformers

In [2]:
labels = {
    "responsavel_execucao": "responsável técnico execução",
    "responsavel_projeto": "responsável técnico projeto",
    "data": "data",
    "endereco_obra": "endereço da obra",
    "proprietario": "proprietário",
    "conteudo": "conteúdo",
    "escala": "escala",
    "desenho": "desenho",
    "assinatura": "assinatura"
}

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
label_texts = list(labels.values())
label_keys = list(labels.keys())

label_embeddings = model.encode(label_texts, normalize_embeddings=True)

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

def classificar(texto):

    emb = model.encode([texto])

    sims = cosine_similarity(emb, label_embeddings)[0]

    idx = np.argmax(sims)

    return label_keys[idx], sims[idx]

In [21]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

labels = {
    "responsavel_execucao": "responsável técnico execução",
    "responsavel_projeto": "responsável técnico projeto",
    "proprietario": "proprietário",
    "conteudo": "conteúdo",
    "endereco": "endereço da obra",
    "data": "data",
    "escala": "escala",
    "desenho": "desenho",
    "assinatura": "assinatura"
}

label_keys = list(labels.keys())
label_embeddings = model.encode(list(labels.values()), normalize_embeddings=True)

def classificar(texto):
    emb = model.encode([texto], normalize_embeddings=True)
    sims = cosine_similarity(emb, label_embeddings)[0]
    idx = np.argmax(sims)
    return label_keys[idx], sims[idx]


THRESHOLD = 0.9

# detectar quais linhas são chaves
chaves = {}

for i, l in enumerate(linhas):
    campo, score = classificar(l)

    if score > THRESHOLD and len(l.split()) <= 4:
        chaves[i] = campo

resultado = {}

indices = list(chaves.keys())

for i, idx in enumerate(indices):

    campo = chaves[idx]

    inicio = idx + 1

    if i + 1 < len(indices):
        fim = indices[i+1]
    else:
        fim = len(linhas)

    valores = linhas[inicio:fim]

    if valores:
        resultado[campo] = valores

print(resultado)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'assinatura': ['PROJETO EXECUTIVO', 'Como indicado', '18/03/2022', 'FAGNER VASCO', '14-39', 'PLANTA BAIXA PAVIMENTO TÉRREO BLOCO B-C', 'SANTIN HOME CLUB', 'Fábio Silva', 'ARQUITETO E URBANISTA | CAU - A131393-2', 'CBA Empreendimentos Imobiliários', 'CPF/CNPJ - 22.407.836./0001-48', 'Rua São Benedito, Serraria - São José']}


In [16]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset

model_name = "microsoft/MiniLM-L12-H384-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=10
)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=32
    )

dataset = Dataset.from_dict({
    "text": [
        "RESPONSÁVEL TÉCNICO EXECUÇÃO",
        "RESP TEC EXEC",
        "ENDEREÇO DA OBRA",
        "DATA",
        "PLANTA BAIXA",
        "18/03/2022"
    ],
    "label": [0,0,1,2,9,9]
})

dataset = dataset.map(tokenize)
dataset.set_format(type="torch", columns=["input_ids","attention_mask","label"])

training_args = TrainingArguments(
    output_dir="./modelo",
    per_device_train_batch_size=8,
    num_train_epochs=20
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/MiniLM-L12-H384-uncased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=20, training_loss=2.157939910888672, metrics={'train_runtime': 7.626, 'train_samples_per_second': 15.736, 'train_steps_per_second': 2.623, 'total_flos': 494116439040.0, 'train_loss': 2.157939910888672, 'epoch': 20.0})

In [17]:
trainer.save_model("modelo_carimbo")
tokenizer.save_pretrained("modelo_carimbo")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('modelo_carimbo/tokenizer_config.json', 'modelo_carimbo/tokenizer.json')

In [18]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "modelo_carimbo"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 384, padding_idx=0)
      (position_embeddings): Embedding(512, 384)
      (token_type_embeddings): Embedding(2, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm): LayerNorm((384,), eps=1e-12,

In [19]:
labels = {
0:"responsavel_execucao",
1:"endereco",
2:"data",
3:"escala",
4:"desenho",
5:"responsavel_projeto",
6:"proprietario",
7:"conteudo",
8:"assinatura",
9:"data"
}

In [20]:
def classificar(texto):

    inputs = tokenizer(
        texto,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=32
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    pred = torch.argmax(logits, dim=1).item()

    return labels[pred]

In [30]:
classificar("19/03/2022")

'data'